[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eirasf/GCED-AA3/blob/main/en/lab5/lab5.ipynb)

# Lab5: Reinforcement learning - Montecarlo and Temporal Difference methods

In this lab we will go deeper into the control methods of reinforcement learning. In particular, we will focus on tabular methods that do not need to have a model of how the environment works. We will distinguish two families: **Montecarlo** methods and **Temporal Difference** methods.

We will work again with [Gym](https://www.gymlibrary.dev/), although this time we will use the [`Blackjack-v1`](https://gymnasium.farama.org/environments/toy_text/blackjack/#blackjack) environment which, as its name indicates, simulates a game of Blackjack.

>
>  *What is Blackjack?*
>
>According to [Wikipedia](https://en.wikipedia.org/wiki/Blackjack): "Blackjack, also called twenty-one, is a card game, typical of casinos, played with one or more English decks of 52 cards without the jokers, which consists of adding a value as close as possible to 21 but without going over. In a casino, each player at the table plays only against the dealer, trying to get a better hand than the dealer. The dealer is subject to fixed rules that prevent him from making decisions about the game. For example, he is required to ask for a card whenever his score is 16 or less, and required to stand if he adds up to 17 or more. The number cards add their value, the face cards add 10 and the Ace is worth 11 or 1, at the player's choice. In the case of the dealer, the Aces are worth 11 as long as he does not go over 21, and 1 otherwise. The best hand is getting 21 with only two cards, that is, with an Ace plus a card of value 10. This hand is known as Blackjack or natural 21. A Blackjack wins over a 21 obtained with more than two cards."

In this case, the OpenGym environment simulates a single-player game against the dealer. In addition to what was described above, you should know that the agent knows the value of the first card of the dealer, but not the rest of the dealer's cards.

Let's load the environment and explore it briefly.

In [ ]:
# If the packages are not installed, you have to run these lines:
#!pip install gymnasium
import gymnasium as gym
import numpy as np
env = gym.make('Blackjack-v1', render_mode='rgb_array')

Examining the properties and methods of the `env` object with `dir()` we can check that there are two variables that give us all the information we need regarding the state:
 - `env.player`: records the cards that the player has.
 - `env.dealer`: records the cards that the dealer has. This information will not be available to the agent.

You can access these variables as `env.get_wrapper_attr('dealer')` and `env.get_wrapper_attr('player')` respectively.

We can also try to show the environment with the render method. Having defined the `render_mode` as `rgb_array`, we can use `matplotlib` to show it. We will define for that a function called `show_environment`.

The list of actions is in `env.action_space`. It is an object of type `Discrete(2)`, which tells us that there are two different actions, identified with 0 and 1.

In [ ]:
env.reset()

print(env.get_wrapper_attr('dealer'))
print(env.get_wrapper_attr('player'))

import matplotlib.pyplot as plt
def show_environment(env:gym.Env) -> None:
    plt.imshow(env.render())
    plt.axis('off')
    plt.show()

# We try the method to show the environment
show_environment(env)

# Show the available actions
actions = env.action_space
print(type(actions))
print(f'There are {actions.n} actions')

We can check that there are two different actions. We are going to execute actions 0 and 1 to check their effect and examine the variables they return.

In [ ]:
env.reset()

print('Situation before:')
print('Dealer:', env.get_wrapper_attr('dealer'))
print('Player:', env.get_wrapper_attr('player'))

# TODO - Change the action to 0 or 1 and run the cell as many times as you need until you identify which action corresponds to 0 and which to 1.
action = ...
obs, reward, terminated, truncated, info = ...

print('Obs:', obs)
print('Reward:', reward)
print('Terminated:', terminated)
print('Truncated:', truncated)
print('Info:', info)

print('Situation after:')
print('Dealer:', env.get_wrapper_attr('dealer'))
print('Player:', env.get_wrapper_attr('player'))

# TODO - Indicate which action is 0 and which is 1.
STICK = ...
HIT = ...

### Problem representation

In addition to identifying the actions, we have been able to check the following:
 - In `obs` there is all the information the agent needs. It is a triple that indicates, in this order, the player's score, the **visible** score of the dealer (only its first card) and a boolean that indicates whether the player has an ace (which is special because it can count 1 or 11).
 - `terminated` is True when the game ends, that is, when the player stands or when he/she asks for a card and goes over 21.
 - `reward` is -1 when the player loses the game (the dealer got a value closer to 21 without going over) and 1 when the player wins (inverse situation). It is 0 in any other case.

Therefore, we can represent the states with the triple that the observation offers us.

In [ ]:
from collections import namedtuple

# We will represent the states as tuples with the following named fields:
#  - player_score: Represents the sum of the values of the received cards
#  - visible_dealer_score: Represents the sum of the visible cards of the dealer
#  - player_has_ace: Boolean that indicates whether the player has an ace
State = namedtuple('State', ['player_score', 'visible_dealer_score', 'player_has_ace'])

# TODO - Complete the function
def get_state(obs:tuple[int, int, int]) -> State:
    ...

# CHECKS - TODO - Verify that the printed state matches what is shown in the image
env.reset()
obs = env.step(STICK)[0]
state = get_state(obs)
print(state)
show_environment(env)
env.reset()

### Absence of an environment model

This problem is non-deterministic, that is, applying an action to a fixed state does not always give the same results. Let's see an example:
> Let's suppose we are in the state $s=$(17, 1, True). This means the following:
> - The dealer has an ace (and another card that the agent cannot see)
> - The player has an ace and a 6.
> In this situation, let's suppose the agent executes the action $a=$STICK. There are two possibilities:
> - The dealer ends up adding more than 17 (but less than 22), so he/she wins. `reward` will be $r=-1$ and the resulting state will be $s'=$(17,X,True) with $X \in [18,21]$.
> - The dealer ends up going over, so the agent wins. `reward` will be $r=1$ and the state will be $s'=$(17,X,True) with $X > 21$.
> In the same way, if the agent chose $a=$HIT, the state and the reward would depend on chance.

In other words, the agent does not have $p(s',r | s,a)$.

## Definition of policies

We are going to define a random policy, that is, one that for each state $s$ gives the same probability $\pi(a|s)$ to any of the two actions. To do so, following what was done in lab 4, we will declare a `numpy.array` that records, for each state-action pair $s,a$, the probability $\pi(a|s)$ (which will always be the same).

In [ ]:
# TODO - Initialize the policy
random_policy = np.zeros(...)
random_policy[:] = ...

# TODO - Retrieve the sample_action function from lab 4 and adapt it so that it returns the number of the action
...

# CHECKS
assert(random_policy[1,1,1,1]==0.5)

We are also going to define a function to visualize the policy comfortably. For each state, it will show us with a color whether the action to take will be STICK or HIT.

In [ ]:
# Function to show the policy
import matplotlib.pyplot as plt
from matplotlib.axes import Axes
from mpl_toolkits.axes_grid1 import make_axes_locatable
from mpl_toolkits.mplot3d import Axes3D
def draw_policy(policy: np.ndarray) -> None:
    def create_figure(with_ace: bool, ax:Axes):
        x_range = np.arange(1, 11)
        y_range = np.arange(11, 22)
        X, Y = np.meshgrid(x_range, y_range)
        Z = policy[Y[::-1],X ,1 if with_ace else 0,1]
        surf = ax.imshow(Z, cmap=plt.get_cmap('Oranges'), vmin=0, vmax=1, extent=[0.5, 10.5, 10.5, 21.5])
        plt.xticks(x_range, ('A', '2', '3', '4', '5', '6', '7', '8', '9', '10'))
        plt.yticks(y_range)
        ax.set_xlabel('Dealer')
        ax.set_ylabel('Player')
        #ax.grid(color='black', linestyle='-', linewidth=1)
        divider = make_axes_locatable(ax)
        cax = divider.append_axes("right", size="5%", pad=0.1)
        cbar = plt.colorbar(surf, ticks=[0, 1], cax=cax)
        cbar.ax.set_yticklabels(['STICK','HIT'])
        cbar.ax.invert_yaxis()

    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(121)
    ax.set_title('Without ace', fontsize=16)
    create_figure(False, ax)
    ax = fig.add_subplot(122)
    ax.set_title('With ace', fontsize=16)
    create_figure(True, ax)
    plt.show()

# TODO - Show the random policy declared before using the function just defined
...

As the policy is random, it is shown with an intermediate color for any state.

### Use of the policy

We are going to adapt the function that generates an episode following a predefined policy, developed in lab 4. For the Montecarlo methods we will need to record the sequence $S_0,A_0,R_1,S_1,A_1,R_2...S_{T-1},A_{T-1},R_T$ of states visited, actions applied and rewards received, so we will have to adapt the implementation.

In [ ]:
# TODO - Retrieve the simulate_episode function from lab 4 and make the necessary modifications so that it works in this problem.
# In addition, the function must now also return a list of tuples with the states visited, actions applied and rewards received.
def simulate_episode(policy:np.ndarray, show_renders:bool=False, verbose:bool=False) -> tuple[float, int, list[tuple[State, int, float]]]:
    ...
    return (G, counter, trajectory)

G, num_steps, trajectory = simulate_episode(random_policy, verbose=True)

print('Trajectory:', trajectory)

### Checking the performance of the policy

Let's do an experiment to evaluate the effectiveness of the random policy. We will simulate 200 episodes, recording the returns obtained in each one.

We will show statistics and plots that can help us get an idea of how well it works.

In [ ]:
# TODO - Retrieve the check_policy function from lab4
# Modify it so that it records and shows the rewards instead of the number of steps
# and use it to test the random policy
def check_policy(policy:np.ndarray, verbose:bool = False) -> None:
    ...

# Check the returns of the random policy using the function just declared
...

You will be able to check that if you play Blackjack following a random policy, the expectation is that a lot of money will be lost.

## Policy evaluation

In order to improve the random policy, in lab 4 we determined the value $v_\pi(s)$ of each state and, based on it, from any state $s$ we chose the action that led us to the state with the greatest return. However, in order to do that from $v_\pi(s)$, it is necessary to know $p(s',r|s,a)$, and that is not the case in this problem. If we do not know which state $s'$ each action $a$ leads to, we cannot choose the action $a$ that gives the most return.

That is why the Montecarlo methods choose to estimate $q_\pi(s,a)$ or, what is the same, the expected return applying the policy $\pi$ of applying action $a$ in state $s$.

Let's implement the following algorithm, but calculating $q_\pi(s,a)$ in addition to $v_\pi(s)$.

![MC prediction](./img/montecarlo-evaluation-algorithm.png)

In [ ]:
GAMMA = 1.0 # In this problem we will not discount the future rewards when calculating the return

def mc_evaluate(policy:np.ndarray, iterations:int = 2000) -> tuple[np.ndarray, np.ndarray]:
    #Initialize
    # TODO - Indicate the shape of the array that stores v(s) for each state s
    vs = np.zeros(...)
    # TODO - Indicate the shape of the array that stores q(s,a) for each pair s,a
    qsa = np.zeros(...)

    # In returns we will have a list of returns (initially empty) for each s,a pair
    returns = []
    # We create the structure
    for i in range(qsa.shape[0]):
        r_i = []
        for j in range(qsa.shape[1]):
            r_j = []
            for k in range(qsa.shape[2]):
                r_k = []
                for l in range(qsa.shape[3]):
                    r_k.append([])
                r_j.append(r_k)
            r_i.append(r_j)
        returns.append(r_i)

    # Loop
    # Main loop of the algorithm (only the number of iterations indicated by parameter) adapting it to calculate q(s,a) following the policy
    for i in range(iterations):
        # TODO - Obtain the values of an episode simulated following the policy
        episode_G, episode_num_steps, episode_trajectory = ...
        G = 0
        remaining_states:list[State] = []
        for state, _, _ in episode_trajectory:
            remaining_states.append(state)

        for state, action, reward in reversed(episode_trajectory):
            remaining_states.pop()
            # TODO - Update G as dictated by the algorithm
            ...
            if state not in remaining_states:
                # TODO - Add G to the returns of this state-action pair (we will calculate all the averages at the end)
                ...

    # Now we calculate the mean returns
    for i in range(qsa.shape[0]):
        for j in range(qsa.shape[1]):
            for k in range(qsa.shape[2]):
                for l in range(qsa.shape[3]):
                    qsa[i,j,k,l] = np.mean(returns[i][j][k][l])
                    vs[i,j,k] += policy[i,j,k,l] * qsa[i,j,k,l]

    return qsa, vs

random_q_values, random_v_values = mc_evaluate(random_policy)

We are going to represent the value of each state in a plot to visualize it comfortably.

In [ ]:
# We are going to represent in a plot v(s) for each state s
from mpl_toolkits import mplot3d

def plot_vs(vs:np.ndarray) -> None:

    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(121, projection='3d')
    ax.set_title('Without ace', fontsize=16)
    x = np.arange(vs.shape[0])
    y = np.arange(vs.shape[1])

    X, Y = np.meshgrid(x, y)
    without_aces = vs[X,Y,0]
    ax.contour3D(X, Y, without_aces, 50, cmap='binary')
    ax.set_xlabel('player')
    ax.set_ylabel('dealer')
    ax.set_zlabel('v(s)');


    ax = fig.add_subplot(122, projection='3d')
    ax.set_title('With ace', fontsize=16)

    x = np.arange(10,vs.shape[0])
    y = np.arange(vs.shape[1])
    X, Y = np.meshgrid(x, y)
    with_aces = vs[X,Y,1]
    ax.contour3D(X, Y, with_aces, 50, cmap='binary')
    ax.set_xlabel('player')
    ax.set_ylabel('dealer')
    ax.set_zlabel('v(s)');

    plt.show()

plot_vs(random_v_values)

In the plots you can see the following:
 - The assigned values are 'irregular', in the sense that contiguous states have disparate values and there is no uniform trend. This effect is due to the fact that $v_\pi(s)$ is being estimated based on the episodes that have gone through state $s$; when the state is visited few times, the estimation is not very reliable. That is why the effect is reduced if we increase the number of iterations.
 - The 'With ace' plot is more irregular than the 'Without ace' one. Reflecting a little, we can realize that the games in which the player has an ace are infrequent, so the estimations are made from a smaller number of samples and, therefore, they are less reliable.
 - Following the random policy, the values are higher the closer the player is to 21.

Let's examine now our estimation of $q_\pi(s,a)$. To do so, let's look only at the value for the states in which the player has cards that add up to 8 (without having an ace).

In [ ]:
print(random_q_values[8,:,0,:])

We can see, for each possible state of the dealer's cards, the value of executing, respectively, the STICK and HIT actions, following afterwards the random policy.

It is to be expected that many of the estimations of $q_\pi(s,a)$ that we have are even less accurate than those of $v_\pi(s)$, due to the low number of samples.


### Policy improvement

Now that we have the value of $q_\pi(s,a)$ (even if it is a bad estimation), we can improve the random policy and get a policy $\pi'\geq\pi$ simply by making a *greedy* selection of the value $q_\pi(s,a)$ for all the actions $a$ available from any state $s$.

Let's implement this policy improvement.

In [ ]:
# TODO - Retrieve the create_greedy_policy function from lab 4 and adapt it to work with q(s,a) instead of v(s)
def create_greedy_policy(q_values: np.ndarray) -> np.ndarray:
    ...

# We calculate the new policy from the q_values that we had approximated for the random policy
improved_policy = create_greedy_policy(random_q_values)
draw_policy(improved_policy)
check_policy(improved_policy)

It can be checked that the new policy obtains better results than the random policy. However, when visualizing it we observe that it is quite 'irregular', which was to be expected knowing the problem that our estimations of $q_\pi(s,a)$ had (which we use as the basis to choose the new policy).

### Successive improvements

In lab 4, we repeated this policy evaluation cycle to obtain $v_\pi(s)$, from which to obtain a better $\pi'$, to re-evaluate it to obtain $v_{\pi'}(s)$ and from it to obtain an even better $\pi''$... and we continued this process while the policy kept improving.

![Policy iteration](./img/policy-iteration.png)

We could try the same process in this case, but there is an important problem.

Suppose our policy decides that for $s=[19, 7, 1]$ the action $a$ to take is STICK. When simulating an episode following that policy, **we will not be taking any sample of the return of using HIT in that state!**. In other words, we will have a good estimation of $q_\pi([19, 7, 1], $STICK$)$, but we will not have an estimation of $q_\pi([19, 7, 1], $HIT$)$. Therefore, we will not be able to calculate which is the best action, that is, $\pi'([19, 7, 1]) = arg\max_{a'} q_\pi([19, 7, 1], a')$ and our policy iteration would be truncated.

This is a manifestation of the **exploration vs. exploitation** problem that Montecarlo methods must face. There are several solutions:
 - **Exploring starts**: Select the first state and the first action randomly (without following the policy) from among those available.
 - **Use of stochastic policies**: keep a level of randomness in the policy so that it explores sometimes (*$\epsilon$-soft* policies).
 - **Off-policy exploration**: use a policy for the simulations different from the one being learned.

In this lab we will use the second solution. We will do the simulations following the *greedy* policy, but it will have a probability $\epsilon$ of taking a random action. In addition, to speed up the calculation, we will do the evaluation and the improvement of the policy in the same algorithm (in a way similar to what happened in the value iteration algorithm seen in lab 4). The algorithm to follow is very similar to the evaluation algorithm seen previously and is described below:

![Montecarlo ES](./img/mc-control-algorithm.png)

In [ ]:
# TODO - Implement the function based on the description of the algorithm (it is very similar to the mc_evaluate function seen above)
def mc_control(num_steps:int=100000, epsilon:float=0.2) -> np.ndarray:
    qsa = np.zeros(...)
    policy = np.zeros(...)
    # We start from the random policy
    policy[:] = 1.0 / NUM_ACTIONS

    ...

    return create_greedy_policy(policy)

# We calculate the policy using an epsilon of 0.4 to guarantee exploration
montecarlo_es_policy = mc_control(epsilon=0.4)
draw_policy(montecarlo_es_policy)
check_policy(montecarlo_es_policy)

The obtained policy is much better, getting close to the optimal policy for this problem. Try different values of `epsilon` or of `num_steps` to try to get an even better policy.

### Optimal policy

As a reference, we are going to apply the policy that is described in the book "Reinforcement Learning: An Introduction" as optimal for this problem (calculated after a high number of iterations of the Montecarlo Exploring Starts algorithm, similar to the one we have applied).

In [ ]:
HIT = [0.0, 1.0]
STICK = [1.0, 0.0]
barto = np.zeros(montecarlo_es_policy.shape)
barto[:] = 0.5

cutoffs0=[16,12,12,11,11,11,16,16,16,16]
cutoffs1=[18,17,17,17,17,17,17,17,18,18]
for i in range(len(cutoffs0)):
    barto[0:cutoffs0[i] + 1, i + 1, 0] = HIT
    barto[cutoffs0[i] + 1:, i + 1, 0] = STICK
    barto[0:cutoffs1[i] + 1, i + 1, 1] = HIT
    barto[cutoffs1[i] + 1:, i + 1, 1] = STICK

draw_policy(barto)
check_policy(barto)

Even with the optimal policy, playing Blackjack is not a good financial decision.

## Congratulations!
You have completed lab 5, in which we have applied a Montecarlo method to approximate the optimal policy for a problem for which we do not have a model.

If you want to go deeper, try to implement the off-policy exploration algorithm seen in the theory class.